In [1]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

from Utils.json_tools import read_formatted_json, write_formatted_json

import Utils.matlab as matlab_utils

matlab_utils = importlib.reload(matlab_utils)
_required_matlab_api = (
    "PhaseSpec",
    "build_raw_interval_table",
    "build_annotated_interval_table",
)
_missing_matlab_api = [name for name in _required_matlab_api if not hasattr(matlab_utils, name)]
if _missing_matlab_api:
    raise ImportError(
        f"Utils.matlab at {matlab_utils.__file__} is missing required parser API: {_missing_matlab_api}. "
        "Sync the updated parser module and restart the kernel."
    )

PhaseSpec = matlab_utils.PhaseSpec
build_raw_interval_table = matlab_utils.build_raw_interval_table
build_annotated_interval_table = matlab_utils.build_annotated_interval_table


In [2]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m12"
date: str | int = "260417"
multi_recording: bool = False
num_of_rec: int = 1
tracking_camera_frame_rate = 120

subfolder_filler = f"{date}_{num_of_rec}"
base_dir = f"{base_dir}/{date}/{subfolder_filler}" if multi_recording else f"{base_dir}/{date}"


In [7]:
file_names = read_formatted_json("./file_names.json")
sync_data_filename: str = file_names["sync_data_filename"]
sync_data = read_formatted_json(f"{base_dir}/data/{sync_data_filename}.json")

protocol_m_path = Path(f"{base_dir}/{date}.m")
session_dir = Path(f"{base_dir}/{date}")
interval_table_output_path = Path(f"{base_dir}/data/interval_table_df_111.csv")

In [4]:
on_list = sync_data["on_list"]
exposure_sampling_number_list_mid = sync_data["exposure_sampling_number_list_mid"]
on_list_frames = [np.searchsorted(exposure_sampling_number_list_mid, on_exposure).tolist() for on_exposure in on_list]

raw_interval_table = build_raw_interval_table(on_list_frames, tracking_camera_frame_rate)
raw_interval_table


,start_frame,end_frame,duration_frames,duration_s,interval_type
0,934,1109,175,1.458333,other
1,1109,1253,144,1.200000,other
2,1253,1409,156,1.300000,other
3,1524,1715,191,1.591667,other
4,1788,2230,442,3.683333,other
...,...,...,...,...,...
65,366857,370514,3657,30.475000,combo
66,370514,372306,1792,14.933333,combo
67,372306,375950,3644,30.366667,combo
68,375950,376114,164,1.366667,other


In [5]:
protocol_m_path

PosixPath('/mnt/senzailab/Kai/#Recording/m12/260417/260417.m')

In [6]:
phase_spec = PhaseSpec([
    "tracking",
    "pictures",
    "combo",
    "rotation",
    "sweep",
    "sweep_hold",
    "flash",
    "static",
])

annotated_interval_table, corrected_interval_table, expected_protocol_table, phase_list = build_annotated_interval_table(
    raw_interval_table,
    protocol_m_path,
    session_dir,
    tracking_camera_frame_rate,
    phase_spec=phase_spec,
)

interval_table = annotated_interval_table.loc[annotated_interval_table["phase_key"].notna()].reset_index(drop=True)
interval_table


,interval_index,start_frame,end_frame,duration_frames,duration_s,interval_type,interval_type_precise,interval_type_protocol,interval_type_resolved,protocol_phase,...,protocol_match_found,protocol_match_method,protocol_ambiguous,phase_family,phase_name,phase_key,stimulus_name,stimulus_color,protocol_note,bg
0,5,2230,2353,123,1.025000,other,other,pictures,pictures,pictures,...,True,fallback_5pct,False,pictures,pictures,pictures_black.png,black.png,"[1, 1, 1]",pictures,[black.png]
1,69,376114,412227,36113,300.941667,baseline,baseline,baseline,baseline,pictures,...,True,ordered_match,False,baseline,baseline,baseline,Sequoia.png,"[0.1, 0.1, 0.1]",pictures,[Sequoia.png]


In [8]:
expected_protocol_table[[
    "expected_index",
    "phase_family",
    "phase_key",
    "expected_duration_s",
    "stimulus_name",
    "protocol_note",
]]


,expected_index,phase_family,phase_key,expected_duration_s,stimulus_name,protocol_note
0,1,pictures,pictures_black.png,10.0,black.png,pictures
1,2,baseline,baseline,1800.0,Sequoia.png,pictures
2,3,pictures,pictures_Sequoia.png,30.0,Sequoia.png,pictures
3,4,combo,rotation_off,15.0,None,combo_rotation_off_01
4,5,combo,rotation_on,30.0,None,combo_rotation_on_01
...,...,...,...,...,...,...
56,57,combo,rotation_on,30.0,None,combo_rotation_on_14
57,58,combo,dot_off,15.0,None,combo_dot_off_14
58,59,combo,dot_on,30.0,None,combo_dot_on_14
59,60,pictures,pictures_black.png,1.0,black.png,pictures


In [9]:
corrected_interval_table[[
    "start_frame",
    "end_frame",
    "interval_type_precise",
    "interval_type_protocol",
    "interval_type_resolved",
    "phase_key",
    "protocol_match_method",
]]


,start_frame,end_frame,interval_type_precise,interval_type_protocol,interval_type_resolved,phase_key,protocol_match_method
0,934,1109,other,<NA>,other,<NA>,unmatched
1,1109,1253,other,<NA>,other,<NA>,unmatched
2,1253,1409,other,<NA>,other,<NA>,unmatched
3,1524,1715,other,<NA>,other,<NA>,unmatched
4,1788,2230,other,<NA>,other,<NA>,unmatched
...,...,...,...,...,...,...,...
65,366857,370514,combo,<NA>,combo,<NA>,unmatched
66,370514,372306,combo,<NA>,combo,<NA>,unmatched
67,372306,375950,combo,<NA>,combo,<NA>,unmatched
68,375950,376114,other,<NA>,other,<NA>,unmatched


In [ ]:
interval_table.to_csv(interval_table_output_path, index=False)
interval_table_output_path
